<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## CMTC: Análisis Transitorio
### T2.3 (parte a) · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Vector de probabilidades de estado π(t)
2. Ecuaciones de Kolmogorov hacia adelante
3. Solución analítica: exponencial matricial
4. Solución numérica: integración ODE
5. Interpretación del comportamiento transitorio
6. Ejemplo: sistema de 2 máquinas paralelas

## Vector de estado π(t)
<div class='defn'>
El <strong>vector de probabilidades de estado</strong> es:

$$\pi(t) = [\pi_0(t), \pi_1(t), \ldots, \pi_{n-1}(t)]$$

donde πᵢ(t) = P(X(t) = i) es la probabilidad de estar en el estado i en el instante t.
</div>

**Condición inicial:** π(0) = π₀ (distribución inicial conocida)

**Normalización:** ∑ᵢ πᵢ(t) = 1 para todo t ≥ 0

## Ecuaciones de Kolmogorov hacia adelante
<div class='defn'>
Las probabilidades de estado evolucionan según:

$$\frac{d\pi(t)}{dt} = \pi(t) \cdot Q$$

o equivalentemente, para cada estado j:

$$\dot{\pi}_j(t) = \sum_{i \neq j} \pi_i(t) \, q_{ij} - \pi_j(t) \, q_j$$
</div>

**Interpretación:**
- Ganancia de probabilidad: flujo de otros estados hacia j (∑ᵢ≠ⱼ πᵢ qᵢⱼ)
- Pérdida de probabilidad: flujo saliente de j (πⱼ qⱼ)

## Solución analítica: exponencial matricial
<div class='defn'>
La solución de dπ/dt = π Q con condición inicial π(0) es:

$$\pi(t) = \pi(0) \cdot e^{Qt}$$

donde $e^{Qt} = \sum_{k=0}^{\infty} \frac{(Qt)^k}{k!}$ es la <strong>exponencial matricial</strong>.
</div>

Computacionalmente, se evalúa con `scipy.linalg.expm(Q*t)`:
```python
from scipy.linalg import expm
pi_t = pi0 @ expm(Q * t)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# Máquina con fallo (α) y reparación (β)
alpha, beta = 0.4, 1.5
Q = np.array([[-alpha, alpha],
              [beta,   -beta]])

pi0 = np.array([1.0, 0.0])   # inicia operativa

tiempos = np.linspace(0, 10, 300)
pi_traj = np.array([pi0 @ expm(Q * t) for t in tiempos])

# Distribución estacionaria teórica
pi_inf = np.array([beta/(alpha+beta), alpha/(alpha+beta)])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tiempos, pi_traj[:, 0], color='#1A7A9A', lw=2, label='π₀(t) — Operativa')
ax.plot(tiempos, pi_traj[:, 1], color='#C8961E', lw=2, label='π₁(t) — En reparación')
ax.axhline(pi_inf[0], ls='--', color='#1A7A9A', alpha=0.5, label=f'π₀∞ = {pi_inf[0]:.3f}')
ax.axhline(pi_inf[1], ls='--', color='#C8961E', alpha=0.5, label=f'π₁∞ = {pi_inf[1]:.3f}')
ax.set_xlabel('Tiempo t'); ax.set_ylabel('Probabilidad')
ax.set_title('Evolución transitoria de π(t) — máquina 2 estados')
ax.legend(); plt.tight_layout(); plt.show()

## Solución numérica: integración ODE

In [ ]:
from scipy.integrate import solve_ivp

def kolmogorov_adelante(t, pi_flat, Q):
    pi = pi_flat.reshape(1, -1)
    dpi = (pi @ Q).flatten()
    return dpi

sol = solve_ivp(
    kolmogorov_adelante,
    t_span=(0, 10),
    y0=pi0,
    args=(Q,),
    dense_output=True,
    max_step=0.05
)

t_ode = np.linspace(0, 10, 300)
pi_ode = sol.sol(t_ode).T

# Comparar ambas soluciones
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tiempos, pi_traj[:, 0], 'b-', lw=2.5, label='Analítica (expm)')
ax.plot(t_ode, pi_ode[:, 0], 'r--', lw=1.5, label='Numérica (ODE)', alpha=0.8)
ax.set_xlabel('Tiempo t'); ax.set_ylabel('π₀(t) — P(operativa)')
ax.set_title('Comparación: solución analítica vs numérica')
ax.legend(); plt.tight_layout(); plt.show()
print(f"Error máximo: {np.max(np.abs(pi_traj[:,0] - pi_ode[:,0])):.2e}")

## Ejemplo: 2 máquinas paralelas
**Estados:** 0 = ambas operativas, 1 = una en reparación, 2 = ambas falladas
**Tasas:** cada máquina falla con α = 0.3; reparación con β = 1.2

In [ ]:
alpha2, beta2 = 0.3, 1.2
Q2 = np.array([
    [-2*alpha2,  2*alpha2,      0      ],
    [  beta2,  -(alpha2+beta2), alpha2  ],
    [    0,      2*beta2,      -2*beta2 ]
])

pi0_2 = np.array([1.0, 0.0, 0.0])
labels = ['Ambas operativas', 'Una en reparación', 'Ambas falladas']
colors = ['#1A7A9A', '#C8961E', '#C0392B']

tiempos2 = np.linspace(0, 15, 400)
pi2_traj = np.array([pi0_2 @ expm(Q2 * t) for t in tiempos2])

fig, ax = plt.subplots(figsize=(11, 4))
for i, (lbl, col) in enumerate(zip(labels, colors)):
    ax.plot(tiempos2, pi2_traj[:, i], color=col, lw=2, label=lbl)
ax.set_xlabel('Tiempo t'); ax.set_ylabel('Probabilidad')
ax.set_title('Evolución transitoria — 2 máquinas paralelas')
ax.legend(); plt.tight_layout(); plt.show()

# Distribución estacionaria
from scipy.linalg import null_space
A = np.vstack([Q2.T, np.ones((1, 3))])
b = np.zeros(4); b[-1] = 1.0
pi_est = np.linalg.lstsq(A, b, rcond=None)[0]
print("Distribución estacionaria:", dict(zip(labels, np.round(pi_est, 4))))

## Conclusiones
- π(t) evoluciona según las ecuaciones de Kolmogorov: dπ/dt = π Q.
- La solución exacta es π(t) = π(0) · e^Qt (exponencial matricial).
- La solución numérica (ODE) da resultados equivalentes con error controlado.
- En el largo plazo, π(t) converge a la distribución estacionaria π∞.
- El análisis transitorio es crucial cuando el horizonte de tiempo es finito o cuando la distribución inicial importa.

**Siguiente parte:** Estado estable, balance detallado y colas M/M/1.